# Bandit Plots: Visualizing $\hat{E}[Y|A]$ over time

Companion notebook to `bandits.ipynb`. This one has no new algorithms — it just **visualizes how the three bandit algorithms' per-arm estimates evolve over time, across 500 Monte Carlo replications**.

**Setup.** Same 5-arm toy problem from the lecture Colab:

| Arm | Name       | True rate |
|-----|------------|-----------|
| 0   | Economic   | 0.10      |
| 1   | Healthcare | 0.12 (winner) |
| 2   | Education  | 0.08      |
| 3   | Climate    | 0.07      |
| 4   | Unity      | 0.11      |

For each algorithm (ε-greedy, UCB, Thompson sampling), we run **500 independent replications of 2000 rounds** and track the running estimate $\hat{\mu}_a(t)$ for each arm at each time step.

**What the plots show.** For each arm $a$ and time $t$, we compute:

- **Centerline**: the mean of $\hat{\mu}_a(t)$ across replications — $E[\hat{\mu}_a(t)]$.
- **Error band**: $\pm \sigma_a(t)$, the standard deviation of $\hat{\mu}_a(t)$ across replications.
- **Dashed line**: the true rate $\mu_a$.

**What you should see.**
- Arms the algorithm pulls a lot → error bands **shrink fast** → centerline **converges to the true rate**.
- Arms the algorithm abandons early → error bands **stay wide** → centerline **stays biased** at whatever random value the algorithm last observed.
- Healthcare (true rate 0.12, the best arm) should converge cleanly across all three algorithms.
- The worst arms (Climate 0.07, Education 0.08) should have the widest error bars because they get few pulls.
- Thompson sampling should concentrate on Healthcare faster than UCB, and both should beat ε-greedy on the tails.

This is the **visual version** of the explore/exploit tradeoff: you can literally see which arms got "learned" and which got ignored.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# 5-arm toy problem (identical to bandits.ipynb)
true_rates = np.array([0.10, 0.12, 0.08, 0.07, 0.11])
message_names = ['Economic', 'Healthcare', 'Education', 'Climate', 'Unity']
n_arms = len(true_rates)

# Monte Carlo parameters
N_REPS = 500   # replications per algorithm
N_STEPS = 2000 # time steps per replication

arm_colors = ['#e74c3c', '#27ae60', '#f39c12', '#3498db', '#9b59b6']  # one per arm
winner = int(np.argmax(true_rates))

print(f'N_REPS = {N_REPS}, N_STEPS = {N_STEPS}')
print(f'True best arm: {message_names[winner]} (arm {winner}, rate {true_rates[winner]:.2f})')
print(f'Expected runtime: ~15-30 seconds total for all 3 algorithms')


## Simulator

A single function that runs one algorithm for one replication and records the per-arm running mean estimate $\hat{\mu}_a(t)$ at every time step.

Three pluggable strategies (selected via the `algo` argument):
- `'eps'`: ε-greedy with ε = 0.1
- `'ucb'`: UCB1 with exploration constant c = 0.3 (matches bandits.ipynb)
- `'thompson'`: Thompson sampling with Beta(1, 1) priors

For each step $t$, we record $\hat{\mu}_a(t) = \text{clicks}_a(t) / \text{impressions}_a(t)$ for arms with at least one pull, and `NaN` for untouched arms (they carry forward the last observed value in the plotting code).


In [ ]:
def run_once(algo, true_rates, n_steps=N_STEPS, epsilon=0.1, ucb_c=0.3, rng=None):
    """Run one replication of one algorithm. Return (n_arms, n_steps) array of
    running-mean estimates mu_hat[a, t] at each step. Untouched arms are NaN."""
    if rng is None:
        rng = np.random.default_rng()
    K = len(true_rates)
    clicks = np.zeros(K)
    impressions = np.zeros(K)

    # Thompson sampling priors
    alpha = np.ones(K)
    beta_ = np.ones(K)

    mu_hat_trace = np.full((K, n_steps), np.nan)

    for t in range(n_steps):
        # Pick arm
        if algo == 'eps':
            if rng.random() < epsilon or impressions.min() == 0:
                arm = rng.integers(K)
            else:
                rates = clicks / np.maximum(impressions, 1)
                arm = int(np.argmax(rates))
        elif algo == 'ucb':
            if impressions.min() == 0:
                arm = int(np.argmin(impressions))
            else:
                rates = clicks / impressions
                bonus = ucb_c * np.sqrt(np.log(t + 1) / impressions)
                arm = int(np.argmax(rates + bonus))
        elif algo == 'thompson':
            samples = rng.beta(alpha, beta_)
            arm = int(np.argmax(samples))
        else:
            raise ValueError(f'unknown algo: {algo}')

        # Pull it
        reward = int(rng.random() < true_rates[arm])
        impressions[arm] += 1
        clicks[arm] += reward
        if algo == 'thompson':
            if reward:
                alpha[arm] += 1
            else:
                beta_[arm] += 1

        # Record running mean estimate for every arm
        pulled_mask = impressions > 0
        mu_hat_trace[pulled_mask, t] = clicks[pulled_mask] / impressions[pulled_mask]

    return mu_hat_trace


def run_mc(algo, n_reps=N_REPS):
    """Run n_reps replications of algo. Return (n_reps, n_arms, n_steps)."""
    traces = np.full((n_reps, n_arms, N_STEPS), np.nan)
    for r in range(n_reps):
        rng = np.random.default_rng(1000 + r)  # reproducible but distinct per rep
        traces[r] = run_once(algo, true_rates, rng=rng)
    return traces


def forward_fill_nan(traces):
    """For each rep and arm, replace leading NaN with the first non-NaN value,
    and any embedded NaN with the previous value. This is what we plot: the
    running estimate 'as of time t' which carries forward between pulls."""
    out = traces.copy()
    n_reps, n_arms_, n_steps = out.shape
    for r in range(n_reps):
        for a in range(n_arms_):
            row = out[r, a]
            # Find first non-NaN
            first = np.argmax(~np.isnan(row)) if (~np.isnan(row)).any() else n_steps
            if first == n_steps:
                continue  # arm never pulled in this rep
            row[:first] = row[first]
            # Forward-fill any interior NaNs
            for t in range(first + 1, n_steps):
                if np.isnan(row[t]):
                    row[t] = row[t - 1]
            out[r, a] = row
    return out


## Run the Monte Carlo sweeps

3 algorithms × 500 reps × 2000 steps. Takes ~15-30 seconds on Colab.


In [ ]:
import time

results = {}
for algo, label in [('eps', 'Epsilon-Greedy'), ('ucb', 'UCB'), ('thompson', 'Thompson Sampling')]:
    t0 = time.time()
    traces = run_mc(algo)
    traces = forward_fill_nan(traces)
    # centerline = mean across reps, band = std across reps
    mu_mean = np.nanmean(traces, axis=0)  # (n_arms, N_STEPS)
    mu_std = np.nanstd(traces, axis=0)    # (n_arms, N_STEPS)
    results[algo] = {'label': label, 'mu_mean': mu_mean, 'mu_std': mu_std}
    print(f'{label:20s} done in {time.time() - t0:5.1f}s')


## Plot function

One figure per algorithm. 5 curves (one per arm). Shaded $\pm\sigma$ bands across replications. Dashed horizontal lines for the true rates. Zoomed to $[0, 0.25]$ on the y-axis so the action near the true rates is visible.


In [ ]:
def plot_algo(mu_mean, mu_std, title, save_path=None):
    t = np.arange(1, N_STEPS + 1)
    fig, ax = plt.subplots(figsize=(11, 6))

    for a in range(n_arms):
        color = arm_colors[a]
        is_winner = (a == winner)
        centerline = mu_mean[a]
        band_lo = centerline - mu_std[a]
        band_hi = centerline + mu_std[a]

        # Shaded band (1 std across reps)
        ax.fill_between(t, band_lo, band_hi, color=color, alpha=0.18)
        # Centerline (mean across reps)
        ax.plot(t, centerline, color=color, linewidth=(3 if is_winner else 1.8),
                label=f'{message_names[a]} (true={true_rates[a]:.2f})',
                zorder=(5 if is_winner else 3))
        # True rate as dashed line
        ax.axhline(true_rates[a], color=color, linestyle='--', linewidth=0.8, alpha=0.7)

    ax.set_xlabel('Time step $t$', fontsize=12)
    ax.set_ylabel(r'Running estimate $\hat{\mu}_a(t)$', fontsize=12)
    ax.set_title(f'{title}\n(centerline = mean across {N_REPS} reps, band = $\\pm$1 std across reps)',
                 fontsize=13)
    ax.set_ylim(0, 0.25)
    ax.set_xlim(1, N_STEPS)
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper right', fontsize=10, framealpha=0.95)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


## Plot 1 of 3 — Epsilon-Greedy

Epsilon-greedy pulls a random arm 10% of the time forever, so every arm (even the losers) gets ~10%/K pulls over the run. That means:

- **Error bars shrink for every arm**, not just the winner — everything gets explored at a steady rate.
- **Healthcare (green)** still gets the bulk of pulls (the other 90% exploits), so its band shrinks fastest and its centerline sits cleanly at the true 0.12.
- **Loser arms** (Climate 0.07, Education 0.08) have wider error bars because they only get random-exploration pulls, but they do converge eventually.
- The price: epsilon-greedy **keeps wasting 10% on pure random pulls forever**, even after it "knows" the answer. Watch how much of the total pull budget ends up on the losers.


In [ ]:
plot_algo(results['eps']['mu_mean'], results['eps']['mu_std'],
          'Epsilon-Greedy ($\\epsilon=0.1$): running estimates over time')


## Plot 2 of 3 — UCB (Upper Confidence Bound)

UCB adds an exploration bonus $c \sqrt{\log t / n_a}$ to each arm's empirical mean. Arms that have been pulled few times get a large bonus and rise to the top of the ranking automatically, so UCB explores without ever flipping a random coin.

What to notice:
- **All arms get pulled enough to converge** — UCB's bonus guarantees every arm is eventually tried enough times for its estimate to stabilize.
- **Healthcare** (true rate 0.12) dominates quickly, and its band collapses onto the true rate.
- **Loser arms** have slightly wider bands than Healthcare but narrower than epsilon-greedy's bands — UCB is smarter about *when* to explore and *how much*.
- UCB's theoretical advantage: regret scales like $O(\log t)$, versus $O(\epsilon \cdot t)$ for epsilon-greedy. You can see the consequence in the bands: UCB's losers spend less total budget, so less of the plot is "wasted" on wrong answers.


In [ ]:
plot_algo(results['ucb']['mu_mean'], results['ucb']['mu_std'],
          'UCB ($c=0.3$): running estimates over time')


## Plot 3 of 3 — Thompson Sampling

Thompson sampling maintains a $\text{Beta}(\alpha_a, \beta_a)$ posterior over each arm's rate and pulls the arm whose sampled draw is highest. Exploration is baked into the sampling step: an arm with high uncertainty (small $\alpha + \beta$) has a fatter posterior, so its sampled draws spread wider and it occasionally wins the argmax even when its mean is low.

What to look for:
- **Healthcare converges tightest of any algorithm.** Thompson concentrates on the winner harder than UCB or ε-greedy, so its estimate of the true 0.12 is sharper.
- **The losers have the widest bands of the three algorithms** — because Thompson abandons them fastest, each individual run has only a handful of pulls on the losers, so the across-rep std stays high. In some runs the algorithm gets a bad early draw and over-invests; in others it barely touches the arm at all.
- **This is Thompson's signature tradeoff**: it buys winner-accuracy by giving up loser-accuracy. If you only care about "which arm is best," that's a great trade. If you care about calibrated estimates for every arm (like Lee & Green's advocacy-group setting where you want a backup wording), it's less appealing — which is why Lee & Green rely on IPW to correct for it.


In [ ]:
plot_algo(results['thompson']['mu_mean'], results['thompson']['mu_std'],
          'Thompson Sampling: running estimates over time')


## Summary

Three algorithms, same problem, three qualitatively different convergence patterns:

| Algorithm | Winner (Healthcare) band | Loser bands | Total pulls wasted |
|-----------|-------------------------|-------------|--------------------|
| ε-greedy | Narrow | **Narrowest** (random exploration) | **Highest** (10% random forever) |
| UCB | Narrow | Narrow | Low (log-bounded) |
| Thompson | **Narrowest** | **Widest** (abandoned fast) | Lowest |

The visual punchline: **there is no free lunch.** ε-greedy gives you calibrated estimates on every arm but wastes exploration forever. Thompson concentrates your pulls on the winner (so your winner estimate is tight) but gives you almost no information about the losers. UCB sits in the middle — good winner convergence, reasonable loser convergence, bounded total regret.

For the HW4 tournament, the tradeoff matters: if the scoring only cares about cumulative reward, Thompson's loser-abandonment is free. If the scoring cares about your final ranking across all arms (identification), ε-greedy's patient exploration is actually useful.
